# 11 — The system as a whole

Two checks that cut across the modalities: how long every stage takes on this laptop, and whether retrieval is doing the work or the model already knew the answers.

**Running it.** Every section below is the evaluation's own code. With `RUN = False`
(the default) nothing is recomputed: the results saved in `data/eval/` are loaded
and shown. Set `RUN = True` in the first code cell to measure again, which
overwrites those files. Each part takes a few minutes; latency is measured once per device (`SA_DEVICE=gpu` or `cpu`).

In [1]:
import os
import sys

sys.path.insert(0, os.path.abspath("") if os.path.basename(os.path.abspath("")) == "notebooks"
                else os.path.join(os.path.abspath(""), "notebooks"))
from eval_common import repo_root  # noqa: E402

REPO_ROOT = repo_root()
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "notebooks"))

# The application's settings (backend/service.py). They are read when the
# pipeline modules are imported, so they are set before anything else.
os.environ.setdefault("SA_EMBEDDER", "bge-small")

# False: show the results saved in data/eval. True: run the evaluation again
# and overwrite them (the run time is given at the top of the notebook).
RUN = False

In [2]:
import json

import pandas as pd

EVAL_DIR = REPO_ROOT / "data" / "eval"


def saved(name):
    """A results file from data/eval."""
    return json.loads((EVAL_DIR / name).read_text(encoding="utf-8"))

## Latency

Times every stage a student waits for, on the machine the project runs on.

The system's central constraint is that it runs on one laptop with no network
call, so the cost of that choice has to be stated rather than implied. This
script measures it end to end: loading a document, chunking, embedding,
building the index, retrieving, answering in both styles, and writing one
quiz question.

Cold and warm are reported separately, because they are different experiences.
The first answer of a session includes loading a 1.5B-parameter model from
disk; every answer after it does not, and the interface tells the student so.

Nothing here changes behaviour: each stage is called exactly as the
application calls it. Times are medians of REPEATS runs, except the cold ones,
which happen once by definition.

In [3]:
# The script's command-line options, as it would have read them.
sys.argv = ['notebook']

In [4]:
import json
import os
import statistics
import sys
import time
from pathlib import Path

ROOT = REPO_ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import faiss  # noqa: E402
import numpy as np  # noqa: E402

from backend.pipeline import quiz, sparse as sparse_module  # noqa: E402
from backend.pipeline.device import get_torch_device, should_use_npu  # noqa: E402
from backend.pipeline.embedder import embed, model_key  # noqa: E402
from backend.pipeline.generator import (CHAT_MODEL_NAME, MODEL_NAME,  # noqa: E402
                                        answer_short, complete, explain)
from backend.pipeline.loader import load_pdf  # noqa: E402
from backend.pipeline.preprocessor import preprocess  # noqa: E402
from backend.pipeline.retriever import retrieve  # noqa: E402

RAW_DIR = ROOT / "data" / "raw"
DECK_DIR = ROOT / "data" / "projects" / "AI"
EVAL_DIR = ROOT / "data" / "eval"
# The application pins SA_DEVICE=cpu (service.py), so that is the run that
# describes what a student waits for; SA_DEVICE=gpu measures the Arc GPU and
# is written to its own file.
OUT_PATH = EVAL_DIR / (f"latency_{os.environ.get('SA_DEVICE', 'gpu')}.json")

REPEATS = 3
TOP_K = 3
PROSE_PDF = RAW_DIR / "Whisper.pdf"
SLIDE_PDF = DECK_DIR / "ai-2-breeding.pdf"
QUESTIONS = [
    "How many hours of audio was Whisper trained on?",
    "What sample rate does Whisper use?",
    "What architecture does Whisper use?",
]

In [5]:
def timed(fn, *args, **kwargs):
    t0 = time.perf_counter()
    result = fn(*args, **kwargs)
    return result, time.perf_counter() - t0

In [6]:
def repeat(fn, *args, **kwargs):
    """(result of the last run, seconds for each run)."""
    seconds, result = [], None
    for _ in range(REPEATS):
        result, elapsed = timed(fn, *args, **kwargs)
        seconds.append(elapsed)
    return result, seconds

In [7]:
def stat(seconds):
    return {"median_seconds": round(statistics.median(seconds), 2),
            "min_seconds": round(min(seconds), 2),
            "max_seconds": round(max(seconds), 2),
            "runs": len(seconds)}

In [8]:
def one(seconds):
    return {"seconds": round(seconds, 2), "runs": 1}

In [9]:
def main():
    # Slide text carries Unicode the Windows console cannot encode.
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass
    device = "npu" if should_use_npu() else get_torch_device()
    print(f"device: {device}   embedder: {model_key()}   model: {MODEL_NAME}\n")
    results = {}

    # Reading documents
    print("reading documents")
    prose_pages, seconds = repeat(load_pdf, str(PROSE_PDF))
    results["load_prose_pdf"] = {**stat(seconds), "file": PROSE_PDF.name,
                                 "pages": len(prose_pages)}
    slide_pages, seconds = repeat(load_pdf, str(SLIDE_PDF), figures="off")
    results["load_slides_text_only"] = {**stat(seconds), "file": SLIDE_PDF.name,
                                        "slides": len(slide_pages)}
    slide_pictures, seconds = repeat(load_pdf, str(SLIDE_PDF), figures="auto")
    results["load_slides_pictures_cached"] = {**stat(seconds),
                                              "file": SLIDE_PDF.name,
                                              "slides": len(slide_pictures)}
    for key in ("load_prose_pdf", "load_slides_text_only",
                "load_slides_pictures_cached"):
        print(f"  {key:<32} {results[key]['median_seconds']:>7.2f}s")

    # Chunking and embedding, per document and for a whole subject
    print("\nchunking, embedding, indexing")
    prose_chunks, seconds = repeat(preprocess, prose_pages, chunking="sentence")
    results["chunk_prose_pdf"] = {**stat(seconds), "chunks": len(prose_chunks)}
    slide_chunks, seconds = repeat(preprocess, slide_pictures)
    results["chunk_slide_deck"] = {**stat(seconds), "chunks": len(slide_chunks)}

    # Cold embedder load is the first embed() call of the process.
    _, cold = timed(embed, [str(c) for c in prose_chunks[:1]])
    results["embed_cold_first_call"] = one(cold)
    _, seconds = repeat(embed, prose_chunks)
    results["embed_prose_chunks"] = {**stat(seconds), "chunks": len(prose_chunks)}

    subject_paths = sorted(DECK_DIR.glob("*.pdf"))
    t0 = time.perf_counter()
    subject_chunks = []
    for path in subject_paths:
        subject_chunks.extend(preprocess(load_pdf(str(path), figures="auto")))
    vectors = embed(subject_chunks)
    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(np.ascontiguousarray(vectors, dtype=np.float32))
    keywords = sparse_module.build_index(subject_chunks)
    results["index_whole_subject"] = {
        **one(time.perf_counter() - t0),
        "documents": len(subject_paths), "chunks": len(subject_chunks),
        "note": "fifteen slide decks, pictures already in the figure cache",
    }
    for key in ("chunk_prose_pdf", "chunk_slide_deck", "embed_cold_first_call",
                "embed_prose_chunks", "index_whole_subject"):
        value = results[key]
        print(f"  {key:<32} "
              f"{value.get('median_seconds', value.get('seconds')):>7.2f}s")

    # Retrieval
    print("\nretrieval")
    _, seconds = repeat(retrieve, QUESTIONS[0], index, subject_chunks, k=TOP_K,
                        sparse=keywords)
    results["retrieve_hybrid"] = {**stat(seconds), "k": TOP_K,
                                  "chunks_searched": len(subject_chunks)}
    print(f"  {'retrieve_hybrid':<32} {results['retrieve_hybrid']['median_seconds']:>7.2f}s")

    # Answering. The first call loads the model, so it is timed on its own.
    print("\nanswering")
    context = [str(c) for c in retrieve(QUESTIONS[0], index, subject_chunks,
                                        k=TOP_K, sparse=keywords)]
    _, cold = timed(answer_short, QUESTIONS[0], context)
    results["answer_short_cold"] = {**one(cold), "note": f"includes loading {MODEL_NAME}"}
    print(f"  {'answer_short_cold':<32} {cold:>7.2f}s")

    seconds = []
    for question in QUESTIONS:
        ctx = [str(c) for c in retrieve(question, index, subject_chunks,
                                        k=TOP_K, sparse=keywords)]
        _, elapsed = timed(answer_short, question, ctx)
        seconds.append(elapsed)
    results["answer_short_warm"] = {**stat(seconds), "questions": len(QUESTIONS)}

    _, cold = timed(explain, QUESTIONS[0], context)
    results["explain_first_call"] = {
        **one(cold),
        "note": ("same weights as the short answer when SA_CHAT_MODEL is unset"
                 if CHAT_MODEL_NAME == MODEL_NAME else f"loads {CHAT_MODEL_NAME}")}
    seconds, words = [], []
    for question in QUESTIONS:
        ctx = [str(c) for c in retrieve(question, index, subject_chunks,
                                        k=TOP_K, sparse=keywords)]
        answer, elapsed = timed(explain, question, ctx)
        seconds.append(elapsed)
        words.append(len(answer.split()))
    results["explain_warm"] = {**stat(seconds), "questions": len(QUESTIONS),
                               "median_words": statistics.median(words)}
    for key in ("answer_short_warm", "explain_first_call", "explain_warm"):
        value = results[key]
        print(f"  {key:<32} "
              f"{value.get('median_seconds', value.get('seconds')):>7.2f}s")

    # One quiz question, written and round-trip checked as the app does it
    print("\nquiz")
    def find(question):
        return retrieve(question, index, subject_chunks, k=TOP_K, sparse=keywords)

    seconds, kept = [], 0
    usable = [c for c in subject_chunks if quiz.usable_for_quiz(c)][:REPEATS * 2]
    for chunk in usable[:REPEATS]:
        item, elapsed = timed(quiz.generate_item, chunk, retrieve_fn=find)
        seconds.append(elapsed)
        kept += item[0] is not None
    results["quiz_item_with_roundtrip"] = {**stat(seconds), "kept": kept,
                                           "attempts": len(seconds)}
    print(f"  {'quiz_item_with_roundtrip':<32} "
          f"{results['quiz_item_with_roundtrip']['median_seconds']:>7.2f}s "
          f"({kept}/{len(seconds)} kept)")

    OUT_PATH.write_text(json.dumps({
        "note": "Wall-clock time per pipeline stage, measured on the project "
                "machine with nothing else running. Cold entries include "
                "loading a model from disk and happen once per process.",
        "device": device,
        "embedder": model_key(),
        "model": MODEL_NAME,
        "chat_model": CHAT_MODEL_NAME,
        "repeats": REPEATS,
        "top_k": TOP_K,
        "stages": results,
    }, indent=2) + "\n", encoding="utf-8")
    print(f"\n  -> {OUT_PATH.relative_to(ROOT)}")

In [10]:
if RUN:
    main()
else:
    print('RUN is False: showing the saved results below.')

RUN is False: showing the saved results below.


### Results

In [11]:
rows = {}
for device in ("gpu", "cpu"):
    lat = saved(f"latency_{device}.json")
    rows[lat["device"]] = {stage: r.get("median_seconds", r.get("seconds"))
                           for stage, r in lat["stages"].items()}
pd.DataFrame(rows).rename_axis("median seconds")

,xpu,cpu
median seconds,,
load_prose_pdf,0.09,0.09
load_slides_text_only,0.20,0.21
load_slides_pictures_cached,5.50,5.68
chunk_prose_pdf,0.15,0.13
chunk_slide_deck,0.00,0.00
embed_cold_first_call,1.43,0.09
embed_prose_chunks,1.07,3.99
index_whole_subject,70.28,72.81
retrieve_hybrid,0.02,0.03


## Does retrieval do the work?

Asks whether retrieval is doing the work.

Every other script here measures the system with retrieval switched on, which
cannot separate two explanations of a correct answer: the passages carried it,
or the model already knew it. Qwen2.5 was trained on the public internet, and
two of the four evaluation papers (Whisper and FLAN) are well known, so this
is not a hypothetical worry.

The same 25 questions are answered three ways, in one process, with the same
model and the same grading rule (generation_analysis.judge, imported rather
than copied so the two sets of numbers stay comparable):

```text
  no_context    — closed book: the question alone, no passages at all.
  retrieved     — the shipped configuration: sentence chunks, bge-small,
                  hybrid retrieval, the top 3 passages.
  wrong_context — the top 3 passages for a DIFFERENT question. A control for
                  the middle case, where any passage at all steadies the
                  model: if this scores like no_context, the gain comes from
                  the right passages rather than from having something to
                  read; if it scores like retrieved, the passages are not
                  what the answer rests on.
```

The short answer style is used throughout, because that is what every recorded
measurement describes.

Differences on 25 questions are small, so retrieved is compared with the other
two by McNemar's exact test, which looks only at the questions whose outcome
changed.

In [12]:
# The script's command-line options, as it would have read them.
sys.argv = ['notebook']

In [13]:
import json
import os
import sys
from math import comb
from pathlib import Path

ROOT = REPO_ROOT
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

import faiss  # noqa: E402
import numpy as np  # noqa: E402

from backend.pipeline import sparse as sparse_module  # noqa: E402
from backend.pipeline.embedder import embed, model_key  # noqa: E402
from backend.pipeline.generator import MODEL_NAME, answer_short  # noqa: E402
from backend.pipeline.loader import load_file  # noqa: E402
from backend.pipeline.preprocessor import preprocess  # noqa: E402
from backend.pipeline.retriever import DENSE_WEIGHT, retrieve  # noqa: E402
from eval_common import judge  # noqa: E402

RAW_DIR = ROOT / "data" / "raw"
EVAL_DIR = ROOT / "data" / "eval"
GROUND_TRUTH_PATH = EVAL_DIR / "retrieval_ground_truth.json"
OUT_PATH = EVAL_DIR / "no_retrieval.json"

DOCUMENTS = [
    "embedding.pdf",
    "Whisper.pdf",
    "Flant5pdf.pdf",
    "Hallucinations_in_Large_Language_Models_LLMs.pdf",
]
CHUNKING = "sentence"
TOP_K = 3
# Which other question's passages the wrong_context arm uses. Coprime with 25,
# so every question is paired with a different one.
OFFSET = 7

In [14]:
def mcnemar(a_correct, b_correct) -> dict:
    """
    McNemar's exact test on paired correct/incorrect outcomes. Only the
    questions where the two arms disagree carry information; under the null
    each is equally likely to fall either way.
    """
    only_a = sum(1 for a, b in zip(a_correct, b_correct) if a and not b)
    only_b = sum(1 for a, b in zip(a_correct, b_correct) if b and not a)
    n = only_a + only_b
    if n == 0:
        return {"only_a": 0, "only_b": 0, "p_value": 1.0}
    tail = sum(comb(n, i) for i in range(min(only_a, only_b) + 1))
    p = min(1.0, 2 * tail / (2 ** n))
    # Three significant figures, not three decimals: a decisive result here is
    # of the order of 1e-5, and rounding it to 0.0 would state something
    # stronger than the test supports.
    return {"only_a": only_a, "only_b": only_b, "p_value": float(f"{p:.3g}")}

In [15]:
def main():
    try:
        sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    except Exception:
        pass

    ground_truth = json.loads(GROUND_TRUTH_PATH.read_text(encoding="utf-8"))
    print(f"{len(ground_truth)} questions   model: {MODEL_NAME}   "
          f"embedder: {model_key()}\n")

    chunks = []
    for name in DOCUMENTS:
        chunks.extend(preprocess(load_file(str(RAW_DIR / name)), chunking=CHUNKING))
    vectors = embed(chunks)
    index = faiss.IndexFlatL2(vectors.shape[1])
    index.add(np.ascontiguousarray(vectors, dtype=np.float32))
    keywords = sparse_module.build_index(chunks)
    print(f"{len(chunks)} chunks across {len(DOCUMENTS)} documents\n")

    passages = [retrieve(entry["question"], index, chunks, k=TOP_K,
                         sparse=keywords, dense_weight=DENSE_WEIGHT)
                for entry in ground_truth]

    results = []
    print(f"{'#':>3}  {'closed':>6} {'top 3':>6} {'wrong':>6}   question")
    for i, entry in enumerate(ground_truth):
        other = passages[(i + OFFSET) % len(ground_truth)]
        arms = {
            "no_context": [],
            "retrieved": [str(c) for c in passages[i]],
            "wrong_context": [str(c) for c in other],
        }
        record = {"index": i + 1, "question": entry["question"],
                  "expected_answer": entry["answer"],
                  "expected_source": {"file": entry["source_file"],
                                      "page": entry["page"]},
                  # Did the control accidentally hand over the right page?
                  "wrong_context_holds_the_source": any(
                      getattr(c, "source_file", None) == entry["source_file"]
                      and getattr(c, "page", None) == entry["page"] for c in other),
                  "arms": {}}
        for arm, context in arms.items():
            answer = answer_short(entry["question"], context)
            correct, signals = judge(entry["answer"], answer)
            record["arms"][arm] = {"answer": answer, "correct": bool(correct),
                                   "signals": signals}
        results.append(record)
        marks = {True: "  yes ", False: "   no "}
        print(f"{i + 1:>3}  {marks[record['arms']['no_context']['correct']]:>6}"
              f"{marks[record['arms']['retrieved']['correct']]:>6}"
              f"{marks[record['arms']['wrong_context']['correct']]:>6}   "
              f"{entry['question'][:58]}")

    arms = ["no_context", "retrieved", "wrong_context"]
    outcomes = {a: [r["arms"][a]["correct"] for r in results] for a in arms}
    totals = {a: sum(outcomes[a]) for a in arms}
    leaked = sum(1 for r in results if r["wrong_context_holds_the_source"])

    print()
    for arm in arms:
        print(f"  {arm:<14} {totals[arm]:>2}/{len(results)}  "
              f"({totals[arm] / len(results):.0%})")
    print(f"\n  the control handed over the right page anyway: {leaked} question(s)")

    tests = {
        "retrieved_vs_no_context": mcnemar(outcomes["retrieved"],
                                           outcomes["no_context"]),
        "retrieved_vs_wrong_context": mcnemar(outcomes["retrieved"],
                                              outcomes["wrong_context"]),
        "wrong_context_vs_no_context": mcnemar(outcomes["wrong_context"],
                                               outcomes["no_context"]),
    }
    for name, t in tests.items():
        print(f"  {name:<30} gained {t['only_a']}, lost {t['only_b']}, "
              f"p = {t['p_value']:.3g}")

    OUT_PATH.write_text(json.dumps({
        "note": "Does retrieval do the work? The same questions answered with "
                "no passages, with the retrieved passages, and with another "
                "question's passages. Same model, same grading rule as "
                "generation_analysis.py. only_a / only_b in the tests are the "
                "questions the first arm gets right and the second does not, "
                "and the other way round; p is McNemar's exact two-sided test.",
        "model": MODEL_NAME,
        "embedder": model_key(),
        "chunking": CHUNKING,
        "k": TOP_K,
        "dense_weight": DENSE_WEIGHT,
        "answer_style": "short",
        "wrong_context_offset": OFFSET,
        "correct": totals,
        "control_leaked_source_page": leaked,
        "tests": tests,
        "results": results,
    }, indent=2, ensure_ascii=False) + "\n", encoding="utf-8")
    print(f"\n  -> {OUT_PATH.relative_to(ROOT)}")

In [16]:
if RUN:
    main()
else:
    print('RUN is False: showing the saved results below.')

RUN is False: showing the saved results below.


### Results

In [17]:
nr = saved("no_retrieval.json")
display(pd.Series(nr["correct"], name=f"correct of {len(nr['results'])}").to_frame())
pd.DataFrame(nr["tests"]).T

,correct of 25
no_context,2
retrieved,18
wrong_context,1


,only_a,only_b,p_value
retrieved_vs_no_context,16.0,0.0,0.000030
retrieved_vs_wrong_context,18.0,1.0,0.000076
wrong_context_vs_no_context,1.0,2.0,1.000000
